# Lesson 03 - Agentic Design Patterns

In this lesson, we explore three foundational design patterns for building effective AI agents:

1. **Clear Agent Instructions** — Crafting precise, role-defining prompts that guide agent behavior
2. **Structured Output with Pydantic Models** — Ensuring agents return predictable, validated data
3. **Single Responsibility Agents** — Designing focused agents that each do one thing well

We'll apply each pattern to a **travel destination recommender** scenario, progressively building a system that can suggest destinations, check availability, and handle logistics.


## Configurazione


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Modello 1: Istruzioni Chiare per l'Agente

Il modello più efficace è anche il più semplice: scrivere istruzioni chiare e dettagliate per il tuo agente.

Buone istruzioni definiscono:
- **Chi** è l'agente (persona e tono)
- **Cosa** dovrebbe fare (responsabilità passo dopo passo)
- **Come** dovrebbe comportarsi (vincoli e stile)

Qui sotto, creiamo un agente concierge di viaggi con istruzioni esplicite che modellano ogni risposta che produce.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Modello 2: Output Strutturato con Modelli Pydantic

Il testo libero è utile per la conversazione, ma i sistemi a valle necessitano di dati strutturati.
Accoppiando i **modelli Pydantic** con una **funzione strumento**, possiamo:

- Definire uno schema preciso per l'output dell'agente
- Validare automaticamente le risposte
- Integrare in modo affidabile i risultati dell'agente nella logica dell'applicazione

La chiave per l'applicazione è passare `response_format` quando eseguiamo l'agente. Questo obbliga il
modello a restituire un oggetto `TravelRecommendations` validato (disponibile in `response.value`)
invece di testo libero. Lo strumento `get_destination_details` restituisce inoltre un
`DestinationRecommendation` tipizzato, così i dati rimangono strutturati dall'inizio alla fine.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## Modello 3: Agenti a Responsabilità Singola

I compiti complessi traggono vantaggio dalla suddivisione del lavoro tra più agenti focalizzati, ciascuno con una singola responsabilità:

- Un **Esperto di Destinazioni** che conosce i luoghi e le disponibilità
- Un **Pianificatore Logistico** che si occupa di voli, hotel e itinerari

Questo rispecchia il principio dell'ingegneria del software della *separazione delle responsabilità* — ogni agente è più facile da testare, mantenere e migliorare in modo indipendente.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Riepilogo

In questa lezione abbiamo applicato tre modelli di design agentici a uno scenario di raccomandazione di viaggi:

| Modello | Idea Chiave | Vantaggio |
|---|---|---|
| **Istruzioni Chiare** | Definire persona, responsabilità e vincoli inizialmente | Comportamento dell'agente coerente e conforme al marchio |
| **Output Strutturato** | Usare modelli Pydantic come formato di risposta | Risultati convalidati e leggibili da macchina |
| **Responsabilità Singola** | Assegnare a ogni agente un compito focalizzato | Più facile da testare, mantenere e combinare |

Questi modelli si combinano naturalmente — puoi unire istruzioni chiare con output strutturato all’interno di un agente a responsabilità singola per costruire sistemi robusti e pronti per la produzione.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
